In [1]:
import torch
import torch.nn as nn
import torch_numopt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.AdaHessian(model=model, lr_init=0.1, n_samples=1)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.08962859213352203
epoch:  1, loss: 0.053369294852018356
epoch:  2, loss: 0.03293620049953461
epoch:  3, loss: 0.03035558946430683
epoch:  4, loss: 0.0302115585654974
epoch:  5, loss: 0.031231725588440895
epoch:  6, loss: 0.0321844257414341
epoch:  7, loss: 0.0324433408677578
epoch:  8, loss: 0.03201637417078018
epoch:  9, loss: 0.031083393841981888
epoch:  10, loss: 0.030043764039874077
epoch:  11, loss: 0.02918863296508789
epoch:  12, loss: 0.02856086939573288
epoch:  13, loss: 0.028208671137690544
epoch:  14, loss: 0.02794688194990158
epoch:  15, loss: 0.02765786647796631
epoch:  16, loss: 0.027304351329803467
epoch:  17, loss: 0.026881473138928413
epoch:  18, loss: 0.02644173614680767
epoch:  19, loss: 0.02600482478737831
epoch:  20, loss: 0.025593895465135574
epoch:  21, loss: 0.02521049790084362
epoch:  22, loss: 0.024853026494383812
epoch:  23, loss: 0.024523725733160973
epoch:  24, loss: 0.0241758543998003
epoch:  25, loss: 0.02379833720624447
epoch:  26, loss

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8865411523770105
Test metrics:  R2 = 0.8880846133963507


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.AdaHessianLS(model=model, lr_init=1)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.39063894748687744
epoch:  1, loss: 0.2322181910276413
epoch:  2, loss: 0.125522181391716
epoch:  3, loss: 0.05592263117432594
epoch:  4, loss: 0.05592263117432594
epoch:  5, loss: 0.05592263117432594
epoch:  6, loss: 0.043950676918029785
epoch:  7, loss: 0.031065018847584724
epoch:  8, loss: 0.031039511784911156
epoch:  9, loss: 0.031039511784911156
epoch:  10, loss: 0.031039511784911156
epoch:  11, loss: 0.03103949874639511
epoch:  12, loss: 0.031039077788591385
epoch:  13, loss: 0.031036144122481346
epoch:  14, loss: 0.03102949447929859
epoch:  15, loss: 0.0310052540153265
epoch:  16, loss: 0.03094576857984066
epoch:  17, loss: 0.03085673600435257
epoch:  18, loss: 0.03074595518410206
epoch:  19, loss: 0.030639199540019035
epoch:  20, loss: 0.0296942088752985
epoch:  21, loss: 0.028226546943187714
epoch:  22, loss: 0.026620645076036453
epoch:  23, loss: 0.023169131949543953
epoch:  24, loss: 0.020245026797056198
epoch:  25, loss: 0.0178848709911108
epoch:  26, loss

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9842134755495354
Test metrics:  R2 = 0.9863292756064277
